In [ ]:
from utils_analysis_by_step_pred import (
    extract_predictions,
    load_prediction_results,
    load_pickle,
    display_preds_for_idx,
    generate_vocab_lists,
)

folder = "eng_fil"
language_code = "tl"

In [ ]:
data = load_prediction_results(folder)

train_vocabs, test_vocabs = await generate_vocab_lists(
    data["train_targets"], data["test_targets"], folder, language_code
)

cache = load_pickle(f"assets/{folder}_detected_words.pkl")

In [ ]:
outputs = await extract_predictions(
    predictions=data,
    train_vocabs=list(train_vocabs),
    test_vocabs=list(test_vocabs),
    cache=cache,
)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 323/323 [00:00<00:00, 1284132.88it/s]


Observation 1: Models do not always learn to use the vocabulary correctly

In [5]:
print(data["train_sources"][48])
print(data["train_targets"][48])

We focus on the issues that are key to our communities including:
Pinagtutulung-tulungan natin ang mga isyung magkakatulad sa ating mga komunidad, kabilang na ang:


In [6]:
display_preds_for_idx(data["predictions"], 0)[48:52]

['"Ang aming mga 4-mont-old mga tiyakin ang hindi-diabetic at hindi-diabetic," he said.',
 '"Ang aming mga 4-mont-old mga tiyakin ang hindi-diabetic at hindi-diabetic," sinabi niya.',
 '"We natulung-tulungan ang mga may-diabetic ang hindi-diabetic," sabaybay-tulungan niya.',
 '"We natulung-tulungan ang mga may-diabetic ang hindi-diabetic," sabaybay-tulungan niya.']

In [7]:
display_preds_for_idx(data["predictions"], 3)[48:52]

['Sinabi ng mga kasinungalingan ng mga Nobel Committee ng Literature, ang mga kasinungalingan ng 2016 Nobel Prize sa Literature, ay hindi naiwala sa loob ng isang taon.',
 'Ang mga kasinungalingan ng mga kasinungalingan ng mga karanasan ng mga karanasan ng mga karanasan ng mga karanasan ng karanasan ng karanasan ng karanasan ng mga karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan ng karanasan',
 'Ang mga kasinungalingan ng mga kasinungalingan ng mga Nobel sa Literature ay hindi natulung-tulungan ang tiyakin ang winning ang 2016 Nobel Prize sa Literature.',
 'Ang mg

In [8]:
display_preds_for_idx(data["predictions"], 9)[48:52]

['Maaari kang mag-asikasugatan sa mga may-ibang security company, ang ADT Corporation.',
 'Maaari kang mag-asikasugatan sa mga may-ibang security company, ang ADT Corporation.',
 'Maaari kang mag-ilawutan ang isang lawsuit laban sa mga may-ibang security company, ang ADT Corporation.',
 'Maaari kang mag-ilaw natulung-tulungan ang mga may-ibang security company, ang ADT Corporation.']

In [9]:
display_preds_for_idx(data["predictions"], 74)[48:52]

['Sinabi ng mga kasinungalingan ng pamahalaan ng Tasmanian, natutukoy sa pagpapatupad ng mga serbisyo ng mga hospital mula sa loob ng halagang AUD$45 milyon.',
 'Sinabi ng mga kasinungalingan ng Tasmanian pamahalaan, ang mga kasinungalingan ay hindi natulung-tulungan ang mga facilities ng mga hospital at mga klinik mula sa pag-uulit ng dagdag ng AUD$45 milyon.',
 'Sinabi ng mga kasinungalingan ng Tasmanian pamahalaan, ang mga kasinungalingan ng mga hospital ay hindi natulung-tulungan sa pag-uusapan ang mga kasinungalingan ng mga kasinungalingan, sa pag-uulit ng dagdag ng AUD$45 milyon.',
 'Sinabi ng mga kasinungalingan ng Tasmanian government, sa pag-ibang dagdag ng AUD$45 milyon, sa pag-ilaw ng mga hospital at mga karanasan ng Tasmanian.']

Observation 2: Models might fail to pick up words; that could be due to noisy training data

In [10]:
# How many examples have words which are completely not generated?
sum([len(output.expected_but_not_predicted) > 0 for output in outputs])

530

In [11]:
print(data["train_sources"][60])
print(data["train_targets"][60])

The United Nations Special Rapporteur for Myanmar said the army was carrying out "mass murder" and called on the world to isolate the junta and halt its access to weapons.
Sinabi ng UN Special Rapporteur para sa Myanmar na ang hukbo ay nagsasagawa ng "mass pagpatay" at nanawagan sa mundo na ihiwalay ang hunta at ihinto ang pag-access nito sa sandata.


In [12]:
print(data["test_sources"][24])
print(data["test_targets"][24])

Late on Sunday, the United States President Donald Trump, in a statement delivered via the press secretary, announced US troops would be leaving Syria.
Gabi na noong Linggo, ang Pangulo ng Estados Unidos na si Donald Trump, sa isang pahayag na inihatid ng kalihim ng pahayagan, ay inanunsiyo na aalis na sa Syria ang mga hukbo ng Estados Unidos.


In [ ]:
display_preds_for_idx(data["predictions"], 24)

NameError: name 'display_preds_for_idx' is not defined

Observation 3: Models can use words they haven't seen in the training data

In [14]:
# How many examples have words which are not expected but generated?
sum([len(output.unexpected_but_predicted) > 0 for output in outputs])

546

In [15]:
print(data["test_sources"][0])
print(data["test_targets"][0])

"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
"Mayroon na tayong 4 na buwang gulang na daga na hindi diabetic na dating diabetic," dagdag niya.


In [16]:
display_preds_for_idx(data["predictions"], 0)[7]

'"We now have 4-month-old mice that are non-diabetic that used to be diabetic," ujarnya.'

In [17]:
display_preds_for_idx(data["predictions"], 0)[42]

'"We ngayon ay nagkaroon ng dalawang-month-old mga sanggol na hindi-diabetic," he said.'

In [18]:
display_preds_for_idx(data["predictions"], 0)[111]

'"We now nagkaroon kami ng isang taon gulang na lalaki," he said.'

In [26]:
detected_langs = [
    (idx, list(filter(lambda item: item[2] > 0.99, output.detected_languages)))
    for (idx, output) in enumerate(outputs)
]

In [33]:
detected_lang_counts = {}
for item in detected_langs:
    for lang in set([word[1] for word in item[1]]):
        if lang in detected_lang_counts:
            detected_lang_counts[lang] += 1
        else:
            detected_lang_counts[lang] = 1

In [38]:
sorted(detected_lang_counts.items(), key=lambda item: -item[1])

[('id', 650),
 ('ceb', 519),
 ('hi', 228),
 ('hmn', 214),
 ('mg', 163),
 ('ht', 147),
 ('ilo', 128),
 ('qu', 113),
 ('eo', 110),
 ('haw', 106),
 ('ro', 101),
 ('jw', 99),
 ('rw', 99),
 ('pt', 96),
 ('sv', 84),
 ('ms', 82),
 ('so', 80),
 ('nso', 77),
 ('fi', 70),
 ('et', 62),
 ('ha', 59),
 ('su', 59),
 ('mt', 53),
 ('az', 50),
 ('gu', 47),
 ('es', 46),
 ('nl', 45),
 ('af', 44),
 ('zh-CN', 44),
 ('yo', 44),
 ('la', 39),
 ('ga', 38),
 ('bg', 36),
 ('bn', 33),
 ('pl', 28),
 ('it', 28),
 ('uz', 27),
 ('yi', 25),
 ('ln', 19),
 ('lus', 19),
 ('sw', 18),
 ('ml', 17),
 ('el', 16),
 ('ka', 16),
 ('mi', 15),
 ('sm', 15),
 ('hu', 14),
 ('ca', 14),
 ('sq', 14),
 ('ru', 14),
 ('vi', 13),
 ('hr', 13),
 ('lv', 13),
 ('ja', 12),
 ('ar', 11),
 ('lg', 11),
 ('fr', 10),
 ('cy', 10),
 ('ku', 10),
 ('tr', 10),
 ('th', 9),
 ('eu', 8),
 ('ko', 8),
 ('xh', 7),
 ('ig', 7),
 ('ay', 7),
 ('cs', 7),
 ('da', 7),
 ('gd', 6),
 ('lt', 6),
 ('om', 4),
 ('uk', 4),
 ('ts', 4),
 ('mr', 4),
 ('sk', 4),
 ('de', 4),
 ('lb', 

Observation 4: Models learn and unlearn patterns based on the order in which they see samples

In [20]:
print(data["train_sources"][38])
print(data["train_targets"][38])

And no lie was found in their mouth; they are blameless" (Revelation 14:4-5).
At sa kanikaniyang bibig ay walang nasumpungang kasinungalingan: sila'y mga walang dungis" (Pahayag 14:4-5).


In [21]:
def find_intervals(indices: list[int]):
    intervals = list[tuple[int, int]]()
    if len(indices) == 0:
        return intervals
    if len(indices) == 1:
        return [(indices[0], indices[0])]
    current = (indices[0], indices[0])
    for item in indices[1:]:
        if item == current[1] + 1:
            current = (current[0], current[1] + 1)
        else:
            intervals.append(current)
            current = (item, item)
    if (len(intervals) == 0) or (intervals[-1] != current):
        intervals.append(current)
    return intervals

In [22]:
list(
    filter(
        lambda item: len(item[1]) > 0,
        [
            (
                i,
                find_intervals(
                    [
                        idx
                        for idx, preds in enumerate(
                            display_preds_for_idx(data["predictions"], i)
                        )
                        if "sinungaling" in preds.lower()
                    ]
                ),
            )
            for i in range(1012)
        ],
    )
)

[(1, [(142, 143), (147, 154)]),
 (3, [(44, 56), (144, 148)]),
 (5, [(146, 146), (148, 148)]),
 (7, [(38, 44), (47, 52), (146, 147), (149, 149), (157, 157)]),
 (9, [(145, 145)]),
 (11, [(139, 139)]),
 (12, [(142, 142), (148, 148)]),
 (13, [(49, 49)]),
 (15, [(48, 50), (52, 52), (140, 147)]),
 (17, [(148, 149)]),
 (19, [(49, 49)]),
 (20, [(152, 153)]),
 (22, [(52, 52)]),
 (23, [(147, 147)]),
 (28, [(48, 48), (144, 144), (147, 148), (151, 154), (157, 157)]),
 (30, [(142, 143), (146, 149), (153, 153)]),
 (31, [(142, 147)]),
 (32, [(145, 145), (147, 147), (152, 152)]),
 (33, [(143, 143), (146, 146), (148, 148), (150, 150), (152, 152)]),
 (34, [(44, 44), (143, 143), (145, 151), (154, 154)]),
 (35, [(142, 144)]),
 (37, [(155, 158)]),
 (38, [(45, 45), (143, 143)]),
 (44, [(55, 55), (149, 151)]),
 (45, [(147, 148), (150, 152), (158, 158)]),
 (51, [(49, 53), (142, 143), (149, 153)]),
 (52, [(49, 50)]),
 (55, [(50, 51)]),
 (56, [(49, 51), (145, 145)]),
 (57, [(43, 45), (47, 50), (52, 52), (140, 1

In [81]:
test_idx = 7

for item in zip(
    data["train_sources"] * 3,
    data["train_targets"] * 3,
    display_preds_for_idx(data["predictions"], test_idx),
    range(300),
):
    print("Idx: ", item[-1])
    print("Train (Src): ", item[0])
    print("Train (Tgt): ", item[1])
    print("Test  (Src): ", data["test_sources"][test_idx])
    print("Test (Pred): ", item[2])
    print("\n\n")

Idx:  0
Train (Src):  These 'important' issues, fed to us through the news, determine what we think about and lay the foundations for what we discuss socially, whether that's on social media or at a dinner party, as well as influencing the focal point for our national narrative, further amplifying their reach.
Train (Tgt):  Ang mga 'importanteng isyu na ito, na ipinagkaloob sa amin sa pamamagitan ng balita, matukoy kung ano ang iniisip natin at inilatag ang mga pundasyon para sa napag-usapan natin sa sosyal, nasa social media man ito o sa isang hapunan, pati na rin ang nakakaimpluwensya sa focal point para sa ating pambansang pagsasalaysay , karagdagang pagpapalakas ng kanilang pag-abot.
Test  (Src):  Siminoff said sales boosted after his 2013 appearance in a Shark Tank episode where the show panel declined funding the startup.
Test (Pred):  Siminoff said sales boosted after his 2013 appearance in a Shark Tank episode where the show panel declined funding the startup.



Idx:  1
Train 